In [15]:
import pandas as pd
import numpy as np

#
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import train_test_split

# Evaluation
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline

from sklearn.compose import ColumnTransformer
import warnings
warnings.filterwarnings("ignore")

In [3]:
df = pd.read_csv("titanic.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Gender,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
df1 = pd.DataFrame({"PassengerId":[1,4],
                   "GDP":[33,45]})
df1

,PassengerId,GDP
0,1,33
1,4,45


In [6]:
df2 = df.copy()
df3 = df2.merge(df1,on="PassengerId",how="left")
df3.head()

,PassengerId,Survived,Pclass,Name,Gender,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,GDP
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,33.0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,NaN
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,NaN
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,45.0
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,NaN


In [8]:
# df["Cabin"].unique()

In [9]:
df.drop(["Name","PassengerId","Ticket","Cabin"],axis=1,inplace=True)

In [10]:
# check missing
df.isna().sum()

Survived      0
Pclass        0
Gender        0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [12]:
df.isna().mean()*100

Survived     0.000000
Pclass       0.000000
Gender       0.000000
Age         19.865320
SibSp        0.000000
Parch        0.000000
Fare         0.000000
Embarked     0.224467
dtype: float64

In [14]:
data  = df.copy()
train_data = df.copy()

In [16]:
# Split data into train and test sets
train_data, test_data = train_test_split(data,test_size=0.2,random_state=42)

# Split train_data into train and validation sets
train_data, val_data = train_test_split(train_data,test_size=0.2,random_state=42)

#### Featuring

In [21]:
data.describe()

,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,0.386083,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.487123,0.836071,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [23]:
cat_featutres = ["Gender","Embarked"] # Onehotencoding, Missing
num_features  = ["Age","Fare"]        # Missing, scaling

numerical_transformer = Pipeline([
    ("imputer",SimpleImputer(strategy="mean")),
    ("scalar",StandardScaler())
])

cat_transformer = Pipeline([
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("encoder",OneHotEncoder())
])


preprocessing = ColumnTransformer([
    ("numerical_trans",numerical_transformer,num_features),
    ("cat_trans",cat_transformer,cat_featutres)
])

pipeline = Pipeline([
    ("preprocessor",preprocessing),
    ("Model",LogisticRegression())
])


In [24]:
pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numerical_trans',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scalar',
                                                                   StandardScaler())]),
                                                  ['Age', 'Fare']),
                                                 ('cat_trans',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder())]),
                                                  ['Gender', 'Embarked'])])),
                ('Model', LogisticRegression())])

In [25]:
train_data.head(2)

,Survived,Pclass,Gender,Age,SibSp,Parch,Fare,Embarked
517,0,3,male,NaN,0,0,24.15,Q
792,0,3,female,NaN,8,2,69.55,S


In [30]:
# Model training
# x = train_data.drop(("Survived"),axis=1)
# y = train_data["Survived"] 
pipeline.fit(train_data.drop(["Survived"],axis=1),train_data["Survived"])

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numerical_trans',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scalar',
                                                                   StandardScaler())]),
                                                  ['Age', 'Fare']),
                                                 ('cat_trans',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder())]),
                                                  ['Gender', 'Embarked'])])),
                ('Model', LogisticRegression())])

In [32]:
x1 = val_data.drop(("Survived"),axis=1)
y1 = val_data["Survived"] 
accuracy =  pipeline.score(x1,y1)
accuracy

0.8041958041958042

In [33]:
x = test_data.drop(("Survived"),axis=1)
y = test_data["Survived"] 
accuracy =  pipeline.score(x,y)
accuracy

0.776536312849162